# SQLibrarian: training data (Kaggle)

Builds the public training data (Spider + Ar-Spider), drops examples that are too long,
optionally cleans your own pairs against the test set, and pushes everything to a
**private** Hugging Face dataset. Cost: free (Kaggle + Hugging Face free tiers).

This notebook holds no logic: every step calls a script from the repo, so the same
code runs on your laptop and here.

**Before you run it:**
1. Settings (right panel): Accelerator **GPU T4 x2** (only section G needs it), Internet **on**, Persistence **Files only**.
2. Add-ons, Secrets: add `HF_TOKEN` (a Hugging Face token with *write* access) and tick it for this notebook.
   If the GitHub repo is private, also add `GITHUB_TOKEN`.
3. Add data: attach your private **Ar-Spider** Kaggle dataset (the Google Drive zip from the
   [Ar-Spider repo](https://github.com/sasmohaimeed/Ar-Spider), uploaded as a Kaggle dataset).
4. Optional, for section G: attach a Kaggle dataset holding your reviewed pairs (`own_pairs_reviewed.jsonl`).

| Section | What it does | Done when |
|---|---|---|
| A | Settings | the cell prints them |
| B | Secrets and Hugging Face login | it prints your HF user name |
| C | Clone the repo and install it | `import sqlibrarian` works |
| D | Find the Ar-Spider files | it lists `tables.json`, `Ar_train_spider.json`, `Ar_dev_spider.json` |
| E | Spider + Ar-Spider → chat examples | it prints the example counts and one prompt |
| F | Drop examples over 2,048 tokens | it prints how many each split dropped |
| G | Own pairs: remove ones too close to the test set (optional) | it prints dropped/kept counts |
| H | Push to the Hub | the dataset page opens with the new configs |

## A. Settings
Change these if needed. Everything else reads from here.

In [ ]:
REPO_URL = "https://github.com/OmarAmir2001/sqlibrarian.git"
BRANCH = "p0-setup"  # the branch that holds the scripts
HF_DATASET_NAME = "sqlibrarian-training-data"  # created as <your-hf-user>/<this>, private
MAX_TOKENS = 2048
TOKENIZER_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

# Folders under /kaggle/input/ (leave OWN_PAIRS_DIR = None to skip section G)
ARSPIDER_DIR = None  # None = search /kaggle/input for it
OWN_PAIRS_DIR = None  # e.g. "/kaggle/input/sqlibrarian-own-pairs"

WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/sqlibrarian-repo"  # not "sqlibrarian": that name would hide the package
print(REPO_URL, BRANCH, HF_DATASET_NAME, MAX_TOKENS, TOKENIZER_MODEL, sep="\n")

## B. Secrets and Hugging Face login
Secrets stay in Kaggle; they never appear in the notebook or its output.

In [ ]:
import os

from huggingface_hub import login, whoami
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
login(os.environ["HF_TOKEN"])
HF_USER = whoami()["name"]
HF_DATASET_REPO = f"{HF_USER}/{HF_DATASET_NAME}"
print("logged in as", HF_USER, "->", HF_DATASET_REPO)

try:
    GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")  # only needed for a private repo
except Exception:
    GITHUB_TOKEN = None

## C. Get the code
Clones the branch and installs the package **without its dependencies** (`--no-deps`):
Kaggle already has PyTorch, transformers, datasets and sentence-transformers built for its GPU,
and installing ours would replace them. Only `sqlglot`, which Kaggle lacks, is added.

In [ ]:
import os
import subprocess

url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
# Step out of the old clone first, so re-running this cell is safe.
os.chdir(WORK)
subprocess.run(["rm", "-rf", REPO_DIR], check=True)
clone = subprocess.run(
    ["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, REPO_DIR],
    capture_output=True,
    text=True,
)
if clone.returncode != 0:
    # Show git's message with the token blanked out, never the raw command.
    message = clone.stderr.replace(GITHUB_TOKEN, "***") if GITHUB_TOKEN else clone.stderr
    raise SystemExit("git clone failed:\n" + message)
# Don't leave the token saved inside the clone's .git/config
subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
%cd $REPO_DIR
!git log --oneline -1

In [ ]:
!pip install -q uv
!uv pip install --system -q --no-deps -e .
!uv pip install --system -q sqlglot

In [ ]:
# Checked in a NEW Python process, like every later step (!python -m ...).
# This notebook's own Python started before the install, so it can't see the package.
import sys

modules = [
    "sqlibrarian.prompt",
    "sqlibrarian.training.spider_data",
    "sqlglot",
    "datasets",
    "transformers",
    "jinja2",
    "sentence_transformers",
]
check = "; ".join(f"import {m}" for m in modules) + "; print('all imports OK')"
subprocess.run([sys.executable, "-c", check], check=True)

## D. Find the Ar-Spider files
Kaggle may unpack the zip into a sub-folder; the script searches the whole folder.

In [ ]:
from pathlib import Path

if ARSPIDER_DIR is None:
    hits = list(Path("/kaggle/input").rglob("tables.json"))
    assert hits, "tables.json not found: attach your Ar-Spider dataset (Add data)"
    ARSPIDER_DIR = str(hits[0].parent)  # the folder that holds tables.json
print("Ar-Spider:", ARSPIDER_DIR)
for name in ["tables.json", "Ar_train_spider.json", "Ar_dev_spider.json"]:
    found = list(Path(ARSPIDER_DIR).rglob(name))
    print(f"  {name:22s}", found[0] if found else "MISSING: set ARSPIDER_DIR in section A")

## E. Spider + Ar-Spider → chat examples
Downloads Spider from the Hub, attaches the right database description to every question
and saves `train` (Spider + Ar-Spider train) and `ar_dev` (Ar-Spider dev, for reporting only, never trained on).

In [ ]:
!python -m sqlibrarian.training.spider_data --arspider-dir "$ARSPIDER_DIR" --out data/training/public

## F. Drop examples that are too long
Counts each example with the model's own tokenizer and chat template; anything over
`MAX_TOKENS` is dropped (cutting it off would train on a broken schema).

In [ ]:
!python -m sqlibrarian.training.filter_length --data data/training/public --out data/training/public_filtered --model "$TOKENIZER_MODEL" --max-tokens $MAX_TOKENS

## G. Own pairs: remove ones too close to the test set (optional)
Your own pairs are generated and validated on your laptop (they need `records_db`).
Here only the slow part runs, on the GPU: embedding every question with multilingual-e5-large and
dropping training questions too similar to a test question. Skipped when `OWN_PAIRS_DIR` is `None`
or the test set is not in the repo yet.

In [ ]:
own_in = Path(OWN_PAIRS_DIR or "/nonexistent") / "own_pairs_reviewed.jsonl"
test_file = Path("data/eval/test_questions.jsonl")
RUN_OWN = own_in.exists() and test_file.exists()
if RUN_OWN:
    !python -m sqlibrarian.training.decontaminate --train "$own_in" --test "$test_file" --out data/training/own_pairs_clean.jsonl
else:
    print(
        "skipped:",
        "no own pairs attached"
        if not own_in.exists()
        else "no data/eval/test_questions.jsonl in the repo yet",
    )

## H. Push to the Hugging Face Hub
Each part becomes a *config* of one **private** dataset: `public` and, when section G ran, `own`.
Every push is a new commit on the Hub, so earlier versions stay available.
The commit message records which repo commit produced the data.

In [ ]:
from datasets import load_dataset, load_from_disk

commit = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True
).stdout.strip()

public = load_from_disk("data/training/public_filtered")
public.push_to_hub(
    HF_DATASET_REPO,
    config_name="public",
    private=True,
    commit_message=f"public: Spider + Ar-Spider, <= {MAX_TOKENS} tokens (repo {commit})",
)
print("pushed public:", {k: len(v) for k, v in public.items()})

if RUN_OWN:
    own = load_dataset("json", data_files="data/training/own_pairs_clean.jsonl")
    own.push_to_hub(
        HF_DATASET_REPO,
        config_name="own",
        private=True,
        commit_message=f"own: decontaminated pairs (repo {commit})",
    )
    print("pushed own:", {k: len(v) for k, v in own.items()})

print(f"https://huggingface.co/datasets/{HF_DATASET_REPO}")